# BenchPress: Is Our Benchmark Novel?

## How it works

[BenchPress](https://github.com/anadim/llm-benchmark-matrix) has an 83-model × 49-benchmark
matrix of published eval scores. The matrix is ~rank 2: most of what distinguishes models
can be captured by just two numbers ("general capability" and "hard novel reasoning").
This means SVD can predict missing scores with ~5% error — like Netflix predicting
movie ratings.

## Why this helps us

We want to build benchmarks that measure something **new**. If BenchPress can predict
our benchmark scores from existing evals, our benchmark is redundant — it's just
measuring the same two things everything else measures.

## The key insight: model fingerprints

BenchPress doesn't know anything about our new benchmark. But it knows a LOT about
each model. From 49 existing benchmarks, each model has a "fingerprint" — its pattern
of scores — that SVD compresses to ~2 numbers.

So when we say "deepseek-r1 scored 71 on our new task", BenchPress can reason:
"gpt-5.2 is usually ~10% stronger than deepseek-r1 on tasks with this pattern...
I'll predict ~80."

- If gpt-5.2 actually scored 82 → BenchPress nailed it → **redundant** (same ranking
  patterns as existing evals)
- If gpt-5.2 actually scored 35 → BenchPress is way off → **novel** (our benchmark
  breaks the usual model rankings, so it's measuring something new)

## The method: Leave-One-Out (LOO)

We run our benchmark on 5 models. Then for each model:
1. **Hide** its score
2. **Add** the other 4 scores as a new column in the 83×50 matrix
3. **Predict** the hidden score using SVD (leveraging the model's fingerprint from 49 existing benchmarks)
4. **Compare** prediction vs actual

We DO have ground truth — we measured all 5 scores! We just hide one at a time to test
if BenchPress can reconstruct it.

## Calibration

| Median LOO error | Meaning | Example |
|:---:|---|---|
| ~3 pts | Redundant — BenchPress predicts it easily | AIME 2025 (math, correlated with other math evals) |
| ~8 pts | Partially novel | ARC-AGI-2 (abstract reasoning) |
| ~12+ pts | Novel — measures something new | |
| ~18 pts | Random noise (upper bound) | |

## Constraint

The models you test on **must be in the BenchPress matrix** (those 83 models).
BenchPress needs their existing scores on 49 benchmarks to build the fingerprint.
The benchmark itself can be anything new — that's the whole point.

## Cost

This notebook is **pure matrix math** — no LLM API calls, $0 cost.
The cost comes from running your benchmark tasks (in the other notebook).

In [ ]:
#| default_exp benchpress

In [ ]:
#| export
import numpy as np
from evaluating_agi.vendor.llm_benchmark_matrix import (
    M_FULL, OBSERVED, N_MODELS, N_BENCH,
    MODEL_IDS, BENCH_IDS, MODEL_NAMES, BENCH_NAMES,
    MODEL_IDX, BENCH_IDX,
    predict_logit_svd_blend,
    evaluation_harness as _eh,
    all_methods as _am,
)

In [ ]:
#| export
def check_novelty(scores: dict[str, float], name='new_benchmark') -> dict:
    """Check if a benchmark is novel or redundant.
    
    scores: {model_id: score_0_to_100} — model IDs must exist in the BenchPress matrix.
    
    Returns LOO prediction errors. Calibration:
      ~3 pts median error → redundant (like AIME, predictable from other math evals)
      ~8 pts             → somewhat novel  
      ~12+ pts           → novel (like ARC-AGI, measures something new)
      ~18 pts            → random noise territory
    """
    # Validate model IDs
    valid = {m: s for m, s in scores.items() if m in MODEL_IDX}
    bad = set(scores) - set(valid)
    if bad: print(f'⚠ Unknown model IDs (skipped): {bad}')
    if len(valid) < 3: raise ValueError(f'Need ≥3 models in BenchPress matrix, got {len(valid)}')

    # LOO: for each model, hide its score, predict from rest
    results = []
    for held_out, actual in valid.items():
        new_col = np.full((N_MODELS, 1), np.nan)
        for m, s in valid.items():
            if m != held_out: new_col[MODEL_IDX[m], 0] = s

        M_aug = np.hstack([M_FULL, new_col])
        _eh.N_BENCH = _am.N_BENCH = N_BENCH + 1
        try:
            pred = predict_logit_svd_blend(M_aug)[MODEL_IDX[held_out], -1]
        finally:
            _eh.N_BENCH = _am.N_BENCH = N_BENCH

        results.append((held_out, actual, pred, abs(pred - actual)))

    errors = [e for *_, e in results]
    med = np.median(errors)

    # Print report
    print(f'  {"Model":<28s} {"Actual":>6s} {"Pred":>6s} {"Err":>5s}')
    for mid, a, p, e in sorted(results, key=lambda x: -x[3]):
        flag = ' ← miss' if e > 10 else ''
        print(f'  {MODEL_NAMES[mid]:<28s} {a:6.1f} {p:6.1f} {e:5.1f}{flag}')
    label = 'redundant' if med < 5 else 'partial' if med < 10 else 'NOVEL' if med < 16 else 'noise'
    print(f'  Median LOO: {med:.1f} pts → {label}')
    return dict(name=name, median_error=med, results=results)


## Controls: sanity check with known benchmarks

Before using this on our own benchmarks, let's verify it works on benchmarks
already in the matrix:

- **AIME 2025** — a math competition eval, highly correlated with other math benchmarks → should be **redundant** (low error)
- **ARC-AGI-2** — abstract reasoning, known to be orthogonal → should be **novel** (high error)
- **Random noise** — no signal at all → maximum error (upper bound)

In [ ]:
def scores_for_existing(bench_id, n=10):
    """Pull scores from the matrix for an existing benchmark (for testing)."""
    j = BENCH_IDX[bench_id]
    all_scores = {MODEL_IDS[i]: M_FULL[i, j] for i in range(N_MODELS) if OBSERVED[i, j]}
    # Pick models with most coverage (best fingerprints for SVD)
    by_coverage = sorted(all_scores, key=lambda m: -OBSERVED[MODEL_IDX[m]].sum())
    return {m: all_scores[m] for m in by_coverage[:n]}

# AIME: should be predictable
check_novelty(scores_for_existing('aime_2025'), 'AIME 2025 (expect: redundant)');


═══════════════════════════════════════════════════════
  AIME 2025 (expect: redundant)
═══════════════════════════════════════════════════════
  Model                           Actual    Pred   Error
  -----------------------------------------------------
  Claude Sonnet 4.5                 87.0    92.1     5.1
  Claude Opus 4.5                   92.8    96.9     4.1
  Gemini 2.5 Pro                    86.7    82.8     3.9
  GPT-5.2                          100.0    96.7     3.3
  Gemini 3 Pro                      95.0    97.9     2.9
  o3 (high)                         88.9    91.4     2.5
  GPT-5                             94.6    92.4     2.2
  Grok 4                            90.6    92.5     1.9
  Claude Opus 4.6                  100.0    99.1     0.9
  Gemini 3.1 Pro                   100.0    99.2     0.8

  Median LOO error: 2.7 pts
  → ⚠ Likely REDUNDANT (BenchPress predicts it easily)
═══════════════════════════════════════════════════════



In [ ]:
# ARC-AGI-2: should be harder to predict
check_novelty(scores_for_existing('arc_agi_2'), 'ARC-AGI-2 (expect: novel)');


═══════════════════════════════════════════════════════
  ARC-AGI-2 (expect: novel)
═══════════════════════════════════════════════════════
  Model                           Actual    Pred   Error
  -----------------------------------------------------
  Gemini 3 Pro                      31.1    48.1    17.0  ← big miss
  GPT-5                              9.9    20.7    10.8  ← big miss
  GPT-5.2                           52.9    43.0     9.9
  Gemini 3.1 Pro                    77.1    67.3     9.8
  Grok 4                            15.9     8.6     7.3
  Claude Opus 4.5                   37.6    30.4     7.2
  Claude Sonnet 4.5                 13.6    17.2     3.6
  Gemini 2.5 Pro                     4.9     7.0     2.1
  o3 (high)                          6.5     7.2     0.7
  Claude Opus 4.6                   68.8    68.4     0.4

  Median LOO error: 7.2 pts
  → ⚡ Partially novel
═══════════════════════════════════════════════════════



In [ ]:
# Random noise: upper bound
rng = np.random.default_rng(42)
rand_models = [m for m in MODEL_IDS if OBSERVED[MODEL_IDX[m]].sum() > 20][:10]
check_novelty({m: float(rng.uniform(10, 90)) for m in rand_models}, 'Random noise (expect: max error)');


═══════════════════════════════════════════════════════
  Random noise (expect: max error)
═══════════════════════════════════════════════════════
  Model                           Actual    Pred   Error
  -----------------------------------------------------
  Claude Sonnet 4.5                 20.2    61.9    41.6  ← big miss
  GPT-5.2                           17.5    52.9    35.3  ← big miss
  GPT-5                             78.7    47.1    31.6  ← big miss
  o4-mini (high)                    45.1    71.6    26.5  ← big miss
  GPT-5.1                           65.8    43.3    22.5  ← big miss
  o3 (high)                         71.9    58.0    13.9  ← big miss
  Claude Opus 4.5                   46.0    32.5    13.5  ← big miss
  Claude 3.7 Sonnet                 88.0    84.7     3.4
  Claude Opus 4                     72.9    70.3     2.5
  Claude Sonnet 4                   70.9    69.1     1.8

  Median LOO error: 18.2 pts
  → ✓ Looks NOVEL (hard to predict from existing evals)

## Available models for our benchmarks

We can only check novelty using models that are in the BenchPress matrix.
Here are the ones we can call via API (OpenAI/Anthropic/Google/Grok):

In [ ]:
# Good picks: mix of cheap + frontier, all with high coverage in the matrix
recommended = [
    ('gpt-4.1-mini',      'cheap,  10 benchmarks'),
    ('gemini-2.5-flash',  'cheap,  25 benchmarks'),
    ('claude-sonnet-4.5', 'mid,    30 benchmarks'),
    ('deepseek-r1',       'mid,    25 benchmarks'),
    ('gpt-5.2',           'frontier, 37 benchmarks'),
]
print('Recommended models for novelty checking:')
print(f'{"BenchPress ID":<25s} {"Notes"}')
print('-' * 55)
for mid, note in recommended:
    print(f'{mid:<25s} {note}')

Recommended models for novelty checking:
BenchPress ID             Notes
-------------------------------------------------------
gpt-4.1-mini              cheap,  10 benchmarks
gemini-2.5-flash          cheap,  25 benchmarks
claude-sonnet-4.5         mid,    30 benchmarks
deepseek-r1               mid,    25 benchmarks
gpt-5.2                   frontier, 37 benchmarks


## Usage: check your own benchmark

```python
# After running your benchmark on 5 models:
check_novelty({
    'gpt-4.1-mini':      35.0,
    'gemini-2.5-flash':  42.0,
    'claude-sonnet-4.5': 68.0,
    'deepseek-r1':       71.0,
    'gpt-5.2':           89.0,
}, 'My Executive Functions Task')
```